# Huấn luyện PyTorch từ CSV đến checkpoint
Chạy các cell theo thứ tự từ trên xuống.

## 1. Đọc CSV và loại dòng trùng
Kiểm tra dòng trùng trên toàn bộ các cột, gồm cả nhãn. Giữ lần xuất hiện đầu tiên của mỗi dòng và loại các bản sao trước khi chia tập, để bản sao giống hệt không xuất hiện ở cả tập train và tập đánh giá. Bảng bên dưới cũng đếm giá trị thiếu trong dữ liệu sau khi loại trùng.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
csv_path = ROOT / 'data' / 'diabetes_prediction_dataset.csv'
raw = pd.read_csv(csv_path)
duplicate_count = int(raw.duplicated().sum())
df = raw.drop_duplicates(keep='first')

print(f'Số dòng ban đầu: {len(raw):,}')
print(f'Số dòng trùng đã loại: {duplicate_count:,}')
print(f'Số dòng sau khi loại trùng: {len(df):,}')
display(df.isna().sum().rename('Số giá trị thiếu').to_frame())

Số dòng ban đầu: 100,000
Số dòng trùng đã loại: 14
Số dòng sau khi loại trùng: 99,986


,Số giá trị thiếu
year,0
gender,0
age,0
location,0
race:AfricanAmerican,0
race:Asian,0
race:Caucasian,0
race:Hispanic,0
race:Other,0
hypertension,0


## 2. Chia tập, tiền xử lý và huấn luyện
Cell này dùng dữ liệu đã loại trùng từ cell trên. Chia train/validation/test theo tỷ lệ 70/15/15 và giữ tỷ lệ nhãn bằng stratify. Các cột số được điền thiếu bằng median rồi chuẩn hóa. Hai cột phân loại được điền bằng giá trị phổ biến nhất rồi mã hóa one-hot. Pipeline học tham số trên train với `fit_transform(xt)`; validation và test dùng lại các tham số đó. Chọn checkpoint theo validation loss, chọn ngưỡng theo validation F1 và đánh giá cuối trên test. Chạy cell này sẽ lưu lại checkpoint và các kết quả huấn luyện. Output huấn luyện bên dưới được lưu từ lần chạy trước.

In [2]:
"""Train a real PyTorch MLP with recorded epoch losses and held-out evaluation."""
import copy
import json
import sys
import time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.base import clone
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(ROOT/'tools/training'))
sys.path.insert(0, str(ROOT/'src'))
from services.torch_model import RiskMLP, TorchPipeline
from train_models import FEATURES, metric_row, choose_threshold


def main():
    torch.manual_seed(42); np.random.seed(42); torch.set_num_threads(2)
    x, y = df[FEATURES], df.diabetes.astype(int)
    xt, xv, yt, yv = train_test_split(x, y, test_size=.30, stratify=y, random_state=42)
    xv, xe, yv, ye = train_test_split(xv, yv, test_size=.5, stratify=yv, random_state=42)
    old = joblib.load(ROOT/'artifacts/diabetes_model.joblib')
    pre = clone(old['pipeline'].named_steps['preprocess'])
    a = torch.tensor(pre.fit_transform(xt), dtype=torch.float32)
    b = torch.tensor(pre.transform(xv), dtype=torch.float32)
    target = torch.tensor(yt.to_numpy(), dtype=torch.float32)
    valid = torch.tensor(yv.to_numpy(), dtype=torch.float32)
    model = RiskMLP(a.shape[1]); optimizer = torch.optim.Adam(model.parameters(), lr=.001)
    criterion = nn.BCEWithLogitsLoss()
    best, patience, history = float('inf'), 0, []
    started = time.perf_counter()
    for epoch in range(1, 61):
        model.train(); order = torch.randperm(len(a)); total = 0.
        for ids in order.split(512):
            optimizer.zero_grad(); loss = criterion(model(a[ids]), target[ids])
            loss.backward(); optimizer.step(); total += loss.item()*len(ids)
        model.eval()
        with torch.inference_mode(): value = criterion(model(b), valid).item()
        history.append({'epoch':epoch, 'train_loss':total/len(a), 'validation_loss':value})
        if value < best-1e-5:
            best = value; state = copy.deepcopy(model.state_dict()); patience = 0
        else: patience += 1
        if epoch % 5 == 0: print(history[-1], flush=True)
        if patience >= 8: break
    checkpoint = ROOT/'artifacts/pytorch_mlp.pt'
    torch.save({'state_dict':state, 'input_dim':a.shape[1]}, checkpoint)
    joblib.dump(pre, ROOT/'artifacts/pytorch_preprocess.joblib')
    pipe = TorchPipeline(checkpoint, pre); threshold = choose_threshold(pipe, xv, yv)
    duration = time.perf_counter()-started
    report = {'architecture':[int(a.shape[1]),64,32,1], 'framework':'PyTorch',
              'seed':42, 'threshold':threshold, 'epochs':len(history),
              'splits':{'train':len(xt),'validation':len(xv),'test':len(xe)},
              'validation':metric_row('PyTorch MLP',pipe,xv,yv,threshold,duration),
              'test':metric_row('PyTorch MLP',pipe,xe,ye,threshold,duration)}
    (ROOT/'artifacts/pytorch_metadata.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    pd.DataFrame(history).to_csv(ROOT/'results/pytorch_training_history.csv',index=False)
    print(json.dumps(report, indent=2), flush=True)


main()


{'epoch': 5, 'train_loss': 0.1103064529813857, 'validation_loss': 0.10737299174070358}


{'epoch': 10, 'train_loss': 0.09875827253499292, 'validation_loss': 0.09522087126970291}


{'epoch': 15, 'train_loss': 0.08968029985246122, 'validation_loss': 0.08688537776470184}


{'epoch': 20, 'train_loss': 0.08638361140719429, 'validation_loss': 0.08472991734743118}


{'epoch': 25, 'train_loss': 0.08405714332141916, 'validation_loss': 0.08292879164218903}


{'epoch': 30, 'train_loss': 0.08245553315592793, 'validation_loss': 0.08220580965280533}


... (3 outputs left out)

{'epoch': 50, 'train_loss': 0.08026366776454481, 'validation_loss': 0.0815211832523346}


{'epoch': 55, 'train_loss': 0.0801739046240742, 'validation_loss': 0.08131329715251923}


{'epoch': 60, 'train_loss': 0.07980413572403137, 'validation_loss': 0.08120355010032654}


{
  "architecture": [
    15,
    64,
    32,
    1
  ],
  "framework": "PyTorch",
  "seed": 42,
  "threshold": 0.39,
  "epochs": 60,
  "splits": {
    "train": 69990,
    "validation": 14998,
    "test": 14998
  },
  "validation": {
    "model": "PyTorch MLP",
    "threshold": 0.39,
    "accuracy": 0.9723963195092679,
    "precision": 0.9735973597359736,
    "recall": 0.6941176470588235,
    "f1": 0.8104395604395604,
    "roc_auc": 0.9768581728399295,
    "pr_auc": 0.8812323056196648,
    "brier": 0.02272495157594296,
    "tn": 13699,
    "fp": 24,
    "fn": 390,
    "tp": 885,
    "train_seconds": 35.49
  },
  "test": {
    "model": "PyTorch MLP",
    "threshold": 0.39,
    "accuracy": 0.9702627016935591,
    "precision": 0.9747995418098511,
    "recall": 0.6674509803921569,
    "f1": 0.792364990689013,
    "roc_auc": 0.9745733011560669,
    "pr_auc": 0.8686875149076798,
    "brier": 0.024250594356935926,
    "tn": 13701,
    "fp": 22,
    "fn": 424,
    "tp": 851,
    "train_seconds

## 3. Giải thích vòng lặp
`train()` bật dropout và `randperm` xáo thứ tự mẫu. `zero_grad()` xóa gradient, `backward()` lan truyền ngược và `step()` cập nhật trọng số. `eval()` tắt dropout khi đo validation, còn `inference_mode()` tắt theo dõi gradient. `deepcopy(state_dict)` giữ checkpoint tốt nhất. Adam dùng learning rate 0,001, tối đa 60 epoch và patience 8.

## 4. Benchmark ML và Deep Learning trên tập test
So sánh ba mô hình scikit-learn và PyTorch MLP trên cùng 14.998 mẫu test (1.275 mẫu dương tính). Mỗi mô hình dùng ngưỡng được chọn trên validation. Cell này đọc số liệu đã lưu, có thể chạy riêng để xem bảng.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

ml = json.loads((root / "artifacts/model_metadata.json").read_text(encoding="utf-8"))
dl = json.loads((root / "artifacts/pytorch_metadata.json").read_text(encoding="utf-8"))

rows = [
    {**entry["test"], "model": entry["test"]["model"] + " (scikit-learn)"}
    for entry in ml["models"]
]
rows.append(dl["test"])
columns = ["model", "threshold", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "fn"]
benchmark = pd.DataFrame(rows)[columns].set_index("model")
display(benchmark.style.format({
    "threshold": "{:.3f}", "accuracy": "{:.2%}", "precision": "{:.2%}",
    "recall": "{:.2%}", "f1": "{:.4f}", "roc_auc": "{:.4f}", "pr_auc": "{:.4f}",
    "fn": "{:d}",
}))


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,fn
model,,,,,,,,
Logistic Regression (scikit-learn),0.800,94.79%,68.22%,72.55%,0.7032,0.9590,0.8071,350
Random Forest (scikit-learn),0.795,97.04%,98.94%,65.88%,0.7910,0.9712,0.8658,435
MLP (scikit-learn),0.385,96.99%,95.88%,67.53%,0.7925,0.9749,0.8704,414
PyTorch MLP,0.390,97.03%,97.48%,66.75%,0.7924,0.9746,0.8687,424


**Nhận xét.** Hai bản MLP có F1 gần như nhau (0,7925 và 0,7924). MLP scikit-learn bỏ sót 414 mẫu dương tính, còn PyTorch MLP bỏ sót 424. Logistic Regression có Recall cao nhất (72,55%) và F1 thấp nhất (0,7032). Random Forest có Accuracy cao nhất (97,04%). Trong app, PyTorch MLP là mô hình chính và MLP scikit-learn dùng để đối chiếu. Các chỉ số phản ánh kết quả trên tập test của project.

## 5. Tích hợp Deep Learning theo mục 7.1 của yêu cầu thầy

**Patient Features → bộ tiền xử lý đã fit → PyTorch MLP → sigmoid → score và nhãn theo ngưỡng validation.**

Phần này dùng lại checkpoint đang được app gọi, không train lại. Chạy cell thiết lập dưới đây rồi chạy các cell 5.1–5.7 theo thứ tự; có thể bắt đầu từ đây nếu chỉ demo service.

Luồng trong MAS: `DataAgent.validate → DLAgent.predict → DiabetesModelService.predict → TorchPipeline.predict_proba`. Kết quả tiếp tục đi vào Rule Agent và Explanation Agent.


In [3]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
import joblib
from IPython.display import display
pd.set_option('display.max_columns', None)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'artifacts/pytorch_mlp.pt').is_file():
    raise FileNotFoundError('Mở notebook từ thư mục final hoặc final/notebooks.')
sys.path.insert(0, str(ROOT / 'src'))
ml_meta = json.loads((ROOT / 'artifacts/model_metadata.json').read_text(encoding='utf-8'))
dl_meta = json.loads((ROOT / 'artifacts/pytorch_metadata.json').read_text(encoding='utf-8'))
preprocess = joblib.load(ROOT / 'artifacts/pytorch_preprocess.joblib')
from services.model_service import DiabetesModelService
from agents.data_agent import validate as validate_patient
from agents.dl_agent import predict as predict_dl
model_service = DiabetesModelService()
print('Model app đang dùng:', model_service.model_name)
print('Notebook:', ROOT / 'notebooks/09_pytorch_training_full.ipynb')


Model app đang dùng: PyTorch MLP
Notebook: C:\DATA\assignment3\final\notebooks\09_pytorch_training_full.ipynb


### 5.1. Kiến trúc mô hình

Mạng MLP nhận vector sau tiền xử lý. Hai lớp ẩn có 64 và 32 neuron; ReLU tạo phi tuyến; Dropout 0,15 giảm overfitting khi train. Lớp cuối trả **logit**, `BCEWithLogitsLoss` dùng trong huấn luyện. Khi suy luận, `sigmoid` chuyển logit thành score trong [0, 1]. `eval()` tắt Dropout và `inference_mode()` tắt theo dõi gradient.

Code gốc: `src/services/torch_model.py` (`RiskMLP`, `TorchPipeline`).


In [4]:
print(model_service.pipeline.model)
print('Kích thước ghi trong artifact:', dl_meta['architecture'])
print('Số tham số:', sum(p.numel() for p in model_service.pipeline.model.parameters()))
print('Chế độ training:', model_service.pipeline.model.training)


RiskMLP(
  (net): Sequential(
    (0): Linear(in_features=15, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.15, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=1, bias=True)
  )
)
Kích thước ghi trong artifact: [15, 64, 32, 1]
Số tham số: 3137
Chế độ training: False


### 5.2. Dữ liệu huấn luyện và thiết lập thực nghiệm

CSV dùng nhãn `diabetes` (0/1). Loại bản ghi trùng trên toàn bộ cột trước khi chia tập. Chia train/validation/test theo 70/15/15, stratify theo nhãn và random_state=42. Chọn checkpoint theo validation loss và ngưỡng theo validation F1; test được giữ riêng để đánh giá cuối.

Nguồn trực tiếp của thí nghiệm là file CSV trong repository, được nhận diện bằng SHA-256 dưới đây. Đường dẫn và hash không thay thế thông tin xuất xứ/giấy phép của nhà cung cấp dataset.


In [5]:
dataset_path = ROOT / ml_meta['dataset']['path']
data_raw = pd.read_csv(dataset_path)
data_clean = data_raw.drop_duplicates()
dataset_hash = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
print('CSV:', dataset_path)
print('SHA-256:', dataset_hash)
print('Khớp metadata:', dataset_hash == ml_meta['dataset']['sha256'])
print('Dòng gốc / sau loại trùng:', len(data_raw), '/', len(data_clean))
display(data_clean.head())
display(data_clean['diabetes'].value_counts().sort_index().rename('Số mẫu').to_frame())
display(pd.Series(dl_meta['splits'], name='Số mẫu').to_frame())
print('Seed:', dl_meta['seed'], '| Epoch đã chạy:', dl_meta['epochs'])
print('Optimizer: Adam; learning_rate=0.001; batch_size=512; patience=8')


CSV: C:\DATA\assignment3\final\data\diabetes_prediction_dataset.csv
SHA-256: 2797de9377339a814ae5a748154c5035d4db5560415f62ffe4133e794e719594
Khớp metadata: True
Dòng gốc / sau loại trùng: 100000 / 99986
   year  gender   age location  race:AfricanAmerican  race:Asian  \
0  2020  Female  32.0  Alabama                     0           0   
1  2015  Female  29.0  Alabama                     0           1   
2  2015    Male  18.0  Alabama                     0           0   
3  2015    Male  41.0  Alabama                     0           0   
4  2016  Female  52.0  Alabama                     1           0   

   race:Caucasian  race:Hispanic  race:Other  hypertension  heart_disease  \
0               0              0           1             0              0   
1               0              0           0             0              0   
2               0              0           1             0              0   
3               1              0           0             0              0   
4 

### 5.3. Tên feature và thứ tự đầu vào

Có 8 feature thô. Service dựng DataFrame theo đúng `model_metadata.json`, nên thứ tự key trong JSON người dùng không quyết định thứ tự tensor. `ColumnTransformer` xuất các cột số trước, sau đó các cột one-hot; thứ tự thực tế sau mã hóa được đọc từ **bộ tiền xử lý đã fit**, không tự liệt kê bằng tay.


In [6]:
raw_feature_order = ml_meta['features']
encoded_feature_order = preprocess.get_feature_names_out().tolist()
display(pd.DataFrame({'Thứ tự': range(1, len(raw_feature_order)+1), 'Feature gốc': raw_feature_order}))
display(pd.DataFrame({'Thứ tự tensor': range(1, len(encoded_feature_order)+1), 'Feature sau xử lý': encoded_feature_order}))
assert len(encoded_feature_order) == dl_meta['architecture'][0]


   Thứ tự          Feature gốc
0       1               gender
1       2      smoking_history
2       3                  age
3       4                  bmi
4       5          hbA1c_level
5       6  blood_glucose_level
6       7         hypertension
7       8        heart_disease
    Thứ tự tensor                 Feature sau xử lý
0               1                          num__age
1               2                          num__bmi
2               3                  num__hbA1c_level
3               4          num__blood_glucose_level
4               5                 num__hypertension
5               6                num__heart_disease
6               7                cat__gender_Female
7               8                  cat__gender_Male
8               9                 cat__gender_Other
9              10      cat__smoking_history_No Info
10             11      cat__smoking_history_current
11             12         cat__smoking_history_ever
12             13       cat__smoking_history_

### 5.4. Tiền xử lý và kiểm soát rò rỉ dữ liệu

- Cột số: điền thiếu bằng median trên train, sau đó StandardScaler.
- Cột phân loại: điền thiếu bằng mode trên train, sau đó OneHotEncoder với handle_unknown='ignore'.
- Train dùng `fit_transform(xt)`; validation/test và app chỉ dùng `transform`.
- App xác thực hồ sơ trước khi dự đoán. Bộ imputer phục vụ pipeline; không có nghĩa giao diện tự chấp nhận hồ sơ thiếu trường bắt buộc.

Cell dưới chỉ gọi transform trên artifact đã fit, không thay đổi tham số hay ghi đè checkpoint. Code train ở `tools/training/train_torch_model.py`.


In [7]:
display(preprocess)
example_features = {name: data_clean.iloc[0][name].item()
                    if hasattr(data_clean.iloc[0][name], 'item') else data_clean.iloc[0][name]
                    for name in raw_feature_order}
validated = validate_patient({'profile': example_features}, model=model_service)
assert validated['status'] == 'ok', validated['errors']
example_frame = pd.DataFrame([validated['validated_features']], columns=raw_feature_order)
example_tensor = preprocess.transform(example_frame)
print('Kích thước trước xử lý:', example_frame.shape)
print('Kích thước sau xử lý:', example_tensor.shape)
display(pd.DataFrame(example_tensor, columns=encoded_feature_order))


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scale', StandardScaler())]),
                                 ['age', 'bmi', 'hbA1c_level',
                                  'blood_glucose_level', 'hypertension',
                                  'heart_disease']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['gender', 'smoking_history'])])
Kích thước trước xử lý: (1, 8)
Kích thước sau xử

### 5.5. Định dạng đầu ra và demo DL Prediction Agent

`DLAgent` nhận `features` đã xác thực, trả output có cấu trúc:

| Trường | Ý nghĩa |
| --- | --- |
| probability | Score sigmoid của model; chưa phải xác suất lâm sàng đã hiệu chỉnh |
| threshold | Ngưỡng chọn trên validation |
| predicted_class | 1 khi score ≥ threshold, ngược lại 0; nhãn mô hình, không phải chẩn đoán |
| model, model_version | Tên và phiên bản service model |
| feature_names | Thứ tự feature gốc |
| feature_impacts | Chênh lệch score khi thay từng đặc trưng bằng baseline; không chứng minh nhân quả |
| second_opinion | Kết quả model scikit-learn đối chiếu |

Cell dùng một dòng CSV để minh họa định dạng và đường gọi service; **không coi ví dụ này là đánh giá trên dữ liệu mới**.


In [8]:
prediction_output = predict_dl({'features': validated['validated_features']}, model=model_service)
print(json.dumps(prediction_output, ensure_ascii=False, indent=2))
assert prediction_output['predicted_class'] == int(prediction_output['probability'] >= prediction_output['threshold'])


{
  "model": "PyTorch MLP",
  "probability": 0.0,
  "threshold": 0.39,
  "predicted_class": 0,
  "risk_level": "low",
  "feature_impacts": [
    {
      "feature": "age",
      "value": "32.0",
      "baseline": "43.0",
      "impact": -0.0
    },
    {
      "feature": "bmi",
      "value": "27.32",
      "baseline": "27.32",
      "impact": 0.0
    },
    {
      "feature": "hbA1c_level",
      "value": "5.0",
      "baseline": "5.8",
      "impact": -0.0
    },
    {
      "feature": "blood_glucose_level",
      "value": "100",
      "baseline": "140.0",
      "impact": -0.0
    },
    {
      "feature": "hypertension",
      "value": "0",
      "baseline": "0.0",
      "impact": 0.0
    },
    {
      "feature": "heart_disease",
      "value": "0",
      "baseline": "0.0",
      "impact": 0.0
    }
  ],
  "model_version": "2.0.0",
  "second_opinion": {
    "model": "MLP",
    "probability": 3.0725452766828205e-06,
    "threshold": 0.385,
    "predicted_class": 0
  },
  "feature_nam

### 5.6. Phiên bản và nhận diện artifact

`model_version` hiện do service công bố. Hash checkpoint và bộ tiền xử lý giúp xác định chính xác các file dùng cho lần demo, vì chuỗi phiên bản thủ công chưa tự thay đổi khi retrain. Phiên bản Agent và schema message là hai khái niệm riêng.


In [9]:
version_rows = []
for relative in ('artifacts/pytorch_mlp.pt', 'artifacts/pytorch_preprocess.joblib', 'artifacts/pytorch_metadata.json'):
    path = ROOT / relative
    version_rows.append({'Artifact': relative, 'SHA-256': hashlib.sha256(path.read_bytes()).hexdigest()})
print('Model version:', prediction_output['model_version'])
print('Ngưỡng model service / metadata:', model_service.threshold, '/', dl_meta['threshold'])
display(pd.DataFrame(version_rows))


Model version: 2.0.0
Ngưỡng model service / metadata: 0.39 / 0.39
                              Artifact  \
0             artifacts/pytorch_mlp.pt   
1  artifacts/pytorch_preprocess.joblib   
2      artifacts/pytorch_metadata.json   

                                             SHA-256  
0  79f58364a1646deb595e7e8f561cf50e7966932c042845...  
1  1f88b7e326060c91aa8f8bf4158be33b31bc136ba711a5...  
2  6cce2f8201914fcf74ef57db68d0d56c0050b4c068450c...  


### 5.7. Chỉ số đánh giá và giới hạn

Các chỉ số dưới đây đọc từ kết quả train đã lưu; cell không đo lại hoặc thay đổi số liệu. Validation dùng chọn mô hình/ngưỡng; test dùng báo cáo kết quả cuối. Accuracy cần đọc cùng Recall, Precision, F1, ROC-AUC, PR-AUC, Brier và confusion matrix vì nhãn dương tính ít hơn nhiều.

**Accuracy cao không chứng minh mô hình đã được kiểm định lâm sàng.** Còn thiếu đánh giá ngoại viện/ngoại miền, hiệu chỉnh xác suất độc lập, đánh giá nhóm và xác nhận chuyên môn. Không dùng output để tự chẩn đoán hoặc quyết định điều trị.


In [10]:
metric_names = ['threshold', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'brier', 'tn', 'fp', 'fn', 'tp']
metrics = pd.DataFrame({split: {key: dl_meta[split][key] for key in metric_names}
                        for split in ('validation', 'test')}).T
display(metrics)
test = dl_meta['test']
display(pd.DataFrame([[test['tn'], test['fp']], [test['fn'], test['tp']]],
                     index=['Thực tế 0', 'Thực tế 1'], columns=['Dự đoán 0', 'Dự đoán 1']))
positive_count = test['tp'] + test['fn']
print(f"Accuracy test: {test['accuracy']:.2%}; Recall: {test['recall']:.2%}; F1: {test['f1']:.4f}.")
print(f"Bỏ sót {test['fn']}/{positive_count} mẫu dương tính trên test.")
print('Đây là đánh giá trên dataset của project, chưa phải bằng chứng kiểm định lâm sàng.')


            threshold  accuracy  precision    recall        f1   roc_auc  \
validation       0.39  0.972396   0.973597  0.694118  0.810440  0.976858   
test             0.39  0.970263   0.974800  0.667451  0.792365  0.974573   

              pr_auc     brier       tn    fp     fn     tp  
validation  0.881232  0.022725  13699.0  24.0  390.0  885.0  
test        0.868688  0.024251  13701.0  22.0  424.0  851.0  
           Dự đoán 0  Dự đoán 1
Thực tế 0      13701         22
Thực tế 1        424        851
Accuracy test: 97.03%; Recall: 66.75%; F1: 0.7924.
Bỏ sót 424/1275 mẫu dương tính trên test.
Đây là đánh giá trên dataset của project, chưa phải bằng chứng kiểm định lâm sàng.
